# EDA: Validation Parquet Datasets

Analyzes the three validation parquet files:
- `val_combined_457_8192.parquet`
- `val_combined_v1v2v3easy9b.parquet`
- `val_combined_v1v2v3hard.parquet`

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("../data")

files = {
    "val_combined_457_8192": DATA_DIR / "val_combined_457_8192.parquet",
    "val_combined_v1v2v3easy9b": DATA_DIR / "val_combined_v1v2v3easy9b.parquet",
    "val_combined_v1v2v3hard": DATA_DIR / "val_combined_v1v2v3hard.parquet",
}

dfs = {name: pd.read_parquet(path) for name, path in files.items()}

## Schema & Volume

In [ ]:
for name, df in dfs.items():
    print(f"=== {name} ===")
    print(f"  Rows:    {len(df):,}")
    print(f"  Columns: {list(df.columns)}")
    for col, dtype in df.dtypes.items():
        print(f"    {col}: {dtype}")
    print()

## Task-level deduplication

Parquets store one row per (task, attempt/sample). Count unique tasks via `extra_info.task_dir`.

In [ ]:
def task_dirs(df):
    if 'extra_info' in df.columns:
        ei = df['extra_info']
        sample = ei.iloc[0]
        if isinstance(sample, dict) and 'task_dir' in sample:
            return ei.apply(lambda x: x.get('task_dir', None))
    return None

for name, df in dfs.items():
    dirs = task_dirs(df)
    if dirs is not None:
        n_unique = dirs.nunique()
        n_rows = len(df)
        print(f"{name}: {n_rows:,} rows, {n_unique:,} unique task_dirs, {n_rows/n_unique:.1f} rows/task avg")
    else:
        print(f"{name}: no task_dir in extra_info")

## extra_info keys

In [ ]:
for name, df in dfs.items():
    if 'extra_info' in df.columns:
        sample = df['extra_info'].iloc[0]
        if isinstance(sample, dict):
            print(f"=== {name} extra_info ===")
            for k, v in sample.items():
                print(f"  {k}: {repr(v)[:100]}")
        print()

## Prompt / response length distribution

In [ ]:
for name, df in dfs.items():
    print(f"=== {name} ===")
    for col in df.columns:
        s = df[col]
        if s.dtype == object:
            lengths = s.dropna().apply(lambda x: len(str(x)))
            print(f"  {col} len: min={lengths.min()}, median={int(lengths.median())}, max={lengths.max()}, mean={int(lengths.mean())}")
        elif np.issubdtype(s.dtype, np.number):
            print(f"  {col}: min={s.min():.4f}, max={s.max():.4f}, mean={s.mean():.4f}, null={s.isna().sum()}")
    print()

## Data source composition

In [ ]:
for name, df in dfs.items():
    print(f"=== {name} ===")
    if 'data_source' in df.columns:
        vc = df['data_source'].value_counts()
        total = len(df)
        for src, cnt in vc.items():
            print(f"  {src}: {cnt:,} ({100*cnt/total:.1f}%)")
    else:
        print("  (no data_source column — inferring from task_dir prefix)")
        dirs = task_dirs(df)
        if dirs is not None:
            # last 2 path components as task name
            basenames = dirs.dropna().apply(lambda x: Path(str(x)).parent.name)
            vc = basenames.value_counts().head(15)
            for src, cnt in vc.items():
                print(f"  {src}: {cnt}")
    print()

## Reward distribution (if present)

In [ ]:
for name, df in dfs.items():
    reward_col = next((c for c in ['reward', 'reward_model', 'score'] if c in df.columns), None)
    if reward_col:
        r = pd.to_numeric(df[reward_col], errors='coerce')
        print(f"{name} [{reward_col}]:")
        print(f"  total={len(r)}, null={r.isna().sum()}")
        print(f"  mean={r.mean():.4f}, std={r.std():.4f}")
        print(f"  ==0: {(r==0).sum()}, ==1: {(r==1).sum()}, in (0,1): {((r>0)&(r<1)).sum()}")
        print(f"  distribution: {r.value_counts().head(10).to_dict()}")
    else:
        print(f"{name}: no reward column")
    print()

## Prompt structure — messages format

In [ ]:
for name, df in dfs.items():
    if 'prompt' not in df.columns:
        print(f"{name}: no prompt column")
        continue
    sample = df['prompt'].iloc[0]
    print(f"=== {name} ===")
    print(f"  prompt type: {type(sample).__name__}")
    if isinstance(sample, list):
        print(f"  messages count: {len(sample)}")
        for i, msg in enumerate(sample[:3]):
            role = msg.get('role', '?') if isinstance(msg, dict) else '?'
            content = str(msg.get('content', msg))[:120] if isinstance(msg, dict) else str(msg)[:120]
            print(f"    [{i}] role={role}: {content!r}")
    elif isinstance(sample, str):
        print(f"  first 200 chars: {sample[:200]!r}")
    print()

## Overlap between datasets

How many task_dirs appear in multiple datasets?

In [ ]:
task_dir_sets = {}
for name, df in dfs.items():
    dirs = task_dirs(df)
    if dirs is not None:
        task_dir_sets[name] = set(dirs.dropna().unique())

names = list(task_dir_sets.keys())
for i, a in enumerate(names):
    for b in names[i+1:]:
        overlap = task_dir_sets[a] & task_dir_sets[b]
        print(f"{a} ∩ {b}: {len(overlap)} tasks overlap")
        print(f"  only in {a}: {len(task_dir_sets[a] - task_dir_sets[b])}")
        print(f"  only in {b}: {len(task_dir_sets[b] - task_dir_sets[a])}")
        print()

if len(names) == 3:
    all3 = task_dir_sets[names[0]] & task_dir_sets[names[1]] & task_dir_sets[names[2]]
    union = task_dir_sets[names[0]] | task_dir_sets[names[1]] | task_dir_sets[names[2]]
    print(f"All 3 overlap: {len(all3)} tasks")
    print(f"Union (total unique tasks): {len(union)} tasks")

## Summary

In [ ]:
print("="*70)
print("SUMMARY")
print("="*70)
for name, df in dfs.items():
    dirs = task_dirs(df)
    n_tasks = dirs.nunique() if dirs is not None else "?"
    n_rows = len(df)
    reward_col = next((c for c in ['reward', 'reward_model', 'score'] if c in df.columns), None)
    if reward_col:
        r = pd.to_numeric(df[reward_col], errors='coerce')
        reward_info = f"reward mean={r.mean():.3f}, pass rate={( r==1).mean():.3f}"
    else:
        reward_info = "no reward col"
    ds_info = df['data_source'].value_counts().to_dict() if 'data_source' in df.columns else "no data_source col"
    print(f"\n{name}")
    print(f"  rows={n_rows:,}  unique_tasks={n_tasks}")
    print(f"  columns={list(df.columns)}")
    print(f"  {reward_info}")
    print(f"  data_source: {ds_info}")